# core

> Fill in a module description here

In [ ]:
#| default_exp core

In [ ]:
#| export
const UNIT_TO_METERS = Dict(
    "meter" => 1.0, "meters" => 1.0, "m" => 1.0,
    "foot" => 0.3048, "feet" => 0.3048, "ft" => 0.3048,
    "kilometer" => 1000.0, "kilomters" => 1000.0, "km" => 1000.0,
    "mile" => 1609.344, "miles" => 1609.344, "mi" => 1609.344,
    "nautical_mile" => 1852.0, "nmi" => 1852.0,
);

In [ ]:
#| export
to_meters(d::Real, unit::AbstractString) = d * UNIT_TO_METERS[unit];

In [ ]:
#| export
"""
calc_resolution(x_coords, y_coords) -> (xres, yres)

Port of `utils.calc_res`: resolution is the coordinate range divided by
`(n - 1)` (the spacing between adjacent pixel centers), not ` range / n`.
"""
function calc_resolution(x_coords::AbstractVector{<:Real}, y_coords::AbstractVector{<:Real})
    w, h = length(x_coords), length(y_coords)
    xres = w > 1 ? (x_coords[end] - x_coords[1]) / (w - 1) : 0.0
    yres = h > 1 ? (y_coords[end] - y_coords[1]) / (h - 1) : 0.0
    return xres, yres        
end;


In [ ]:
#| export
"""
calc_cellsize(xres, yres; unit="meter") -> (cellsize_x, cellsize_y)
for situations where the pixel resolution is already know. Converts to meters and
takes the `abs` of y resolution , since raster y-resolution is conventionally
negative 
"""
function cal_cellsize(xres::Real, yres::Real; unit::AbstractString="meter")
    cx = to_meters(xres, unit)
    cy = to_meters(yres, unit)
    return abs(cx), abs(cy)    
end;

In [ ]:
#| export
"""
ellipse_kernel(half_w, half_h) -> Matrix{Float64}

Port of `convolution._ellipse_kernel`: a boolean ellipse mask of shape
`(2*half_h+1, 2*half_w+1)` using the implicit equation
`(x*half_h)^2 + (y*half_w)^2 <= (half_w*half_h)^2`, which avoids the
division (and its rounding issues) that a naive `(x/a)^2+(y/b)^2<=1`
form would need.
"""

function ellipse_kernel(half_w::Integer, half_h::Integer)
    xs = -half_w:half_w
    ys = -half_h:half_h
    kernel = zeros(Float64, length(ys), length(xs))
    
    # Cache squares to avoid redundant calculations inside the loop
    w2 = Float64(half_w)^2
    h2 = Float64(half_h)^2
    r2 = w2 * h2
    
    # Outer loop = Columns (j, x), Inner loop = Rows (i, y) for Column-Major efficiency
    @inbounds for (j, x) in enumerate(xs)
        x_term = Float64(x)^2 * h2
        for (i, y) in enumerate(ys)
            y_term = Float64(y)^2 * w2
            kernel[i, j] = (x_term + y_term <= r2) ? 1.0 : 0.0
        end
    end
    
    return kernel
end;

In [ ]:
#| export
"""
circle_kernel(cellsize_x, cellsize_y, radius) -> Matrix{Float64}

Port of `convolution.circle_kernel`. `radius` is a ground distance
(same unit as the cell sizes); it is converted to a pixel half-width
and half-height by dividing by cell size, so non-square pixels
(`cellsize_x != cellsize_y`) give an ellipse in pixel space that is a
true circle on the ground.
"""
function circle_kernel(cellsize_x::Real, cellsize_y::Real, radius::Real)
    radius <= 0 && throw(ArgumentError("radius must be positive, got $radius"))
    
    # Use ceil to ensure the kernel bounding box covers the full physical radius
    half_w = ceil(Int, radius / cellsize_x)
    half_h = ceil(Int, radius / cellsize_y)
    
    xs = -half_w:half_w
    ys = -half_h:half_h
    kernel = zeros(Float64, length(ys), length(xs))
    
    r2 = Float64(radius) ^ 2
    
    # Process column-major order for speed, evaluating physical metric distances
    @inbounds for (j, x_idx) in enumerate(xs)
        # Convert pixel index to actual ground distance (meters)
        dx2 = (Float64(x_idx) * cellsize_x) ^ 2
        
        for (i, y_idx) in enumerate(ys)
            # Convert pixel index to actual ground distance (meters)
            dy2 = (Float64(y_idx) * cellsize_y) ^ 2
            
            # True spatial Pythagorean check (dx² + dy² <= r²)
            kernel[i, j] = (dx2 + dy2 <= r2) ? 1.0 : 0.0
        end
    end
    return kernel
end;

In [ ]:
#| export
"""
annulus_kernel(cellsize_x, cellsize_y, outer_radius, inner_radius) -> Matrix{Float64}

Port of `convolution.annulus_kernel`: a ring-shaped kernel, the outer
circle kernel minus the (zero-padded, centered) inner circle kernel.
This is the shape BPI is built on in bpi.jl -- cells strictly between
`inner_radius` and `outer_radius` of the center, in ground distance.
"""
function annulus_kernel(cellsize_x::Real, cellsize_y::Real, outer_radius::Real, inner_radius::Real)
    outer_radius <= 0 && throw(ArgumentError("outer_radius must be positive, got $outer_radius"))
    inner_radius < 0 && throw(ArgumentError("inner_radius cannot be negative, got $inner_radius"))
    inner_radius >= outer_radius && throw(ArgumentError("inner_radius must be smaller than outer_radius"))
    
    # Calculate bounding box based on the outer radius to fit the whole shape
    half_w = ceil(Int, outer_radius / cellsize_x)
    half_h = ceil(Int, outer_radius / cellsize_y)
    
    xs = -half_w:half_w
    ys = -half_h:half_h
    kernel = zeros(Float64, length(ys), length(xs))
    
    # Pre-square the physical boundaries to speed up the loop math
    r_outer2 = Float64(outer_radius)^2
    r_inner2 = Float64(inner_radius)^2
    
    # Single-pass column-major evaluation
    @inbounds for (j, x_idx) in enumerate(xs)
        dx2 = (Float64(x_idx) * cellsize_x)^2
        
        for (i, y_idx) in enumerate(ys)
            dy2 = (Float64(y_idx) * cellsize_y)^2
            dist2 = dx2 + dy2
            
            # The point is active ONLY if it is inside the outer circle AND outside the inner circle
            kernel[i, j] = (r_inner2 < dist2 <= r_outer2) ? 1.0 : 0.0
        end
    end
    
    return kernel
end;

In [ ]:
#| export
function kernel_mean_diff(data::AbstractMatrix{Float64}, kernel::AbstractMatrix{Float64})
    rows, cols = size(data)
    krows, kcols = size(kernel)
    hr, hc = krows ÷ 2, kcols ÷ 2
    
    valid = .!isnan.(data)
    filled = ifelse.(valid, data, 0.0)

    out = similar(data)
    
    # Outer loop = Columns (j), Inner loop = Rows (i) for extreme speed
    Threads.@threads for j in 1:cols
        @inbounds for i in 1:rows
            # If the center cell itself is NaN, don't bother calculating neighbors
            if !valid[i, j]
                out[i, j] = NaN
                continue
            end
            
            s, c = 0.0, 0.0
            
            # Kernel loop: process row offsets fastest (ki)
            for kj in -hc:hc
                nj = j + kj
                if 1 <= nj <= cols
                    k_col = kj + hc + 1
                    for ki in -hr:hr
                        w = kernel[ki + hr + 1, k_col]
                        w == 0.0 && continue
                        
                        ni = i + ki
                        if 1 <= ni <= rows
                            s += w * filled[ni, nj]
                            c += w * Float64(valid[ni, nj])
                        end
                    end
                end
            end
            out[i, j] = c > 0.0 ? data[i, j] - (s / c) : NaN
        end
    end
    return out
end;


In [ ]:
#| export
"""
    _ray_directions(n_directions, cellsize_x, cellsize_y) -> (dcol, drow, step_m)

Unit step vectors in pixel-index space for `n_directions` evenly spaced ground
azimuths (clockwise from east, since rows increase southwards), plus the ground
length in metres of one unit step along each ray. Callers must use
`r * step_m[d]` as the ground distance and cap rays at
`floor(radius_m / step_m[d])` steps for an isotropic search window.
"""
function ray_directions(n_directions::Integer, cellsize_x::Real, cellsize_y::Real)
    n_directions > 0 && iseven(n_directions) ||
        throw(ArgumentError("n_directions must be a positive even integer (got $n_directions)"))
    cx, cy = abs(float(cellsize_x)), abs(float(cellsize_y))
    dcol   = Vector{Float64}(undef, n_directions)
    drow   = Vector{Float64}(undef, n_directions)
    step_m = Vector{Float64}(undef, n_directions)
    for d in 0:n_directions-1
        t = 2d / n_directions                 # azimuth in units of π
        u = cospi(t) / cx
        v = sinpi(t) / cy
        nrm = hypot(u, v)
        dcol[d+1]   = u / nrm
        drow[d+1]   = v / nrm
        step_m[d+1] = 1 / nrm                 # metres per unit pixel step
    end
    return dcol, drow, step_m
end


In [ ]:
#| export
"""
    _max_elevation_angle(data, y, x, dcol, drow, n_steps, cellsize_x, cellsize_y)
        -> (angle_rad, steps_done)

Largest unclamped elevation angle along one ray (nearest-neighbour sampling),
and the number of valid steps walked before hitting the raster edge, a NaN
cell, or `n_steps`. Returns `(NaN, 0)` for a NaN centre or an empty ray, so
the caller can decide whether the cell has enough coverage to be reported.
"""
function max_elevation_angle(data::AbstractMatrix{T}, y::Integer, x::Integer,
                              dcol::Real, drow::Real, n_steps::Integer,
                              cellsize_x::Real, cellsize_y::Real) where {T<:AbstractFloat}
    rows, cols = size(data)
    center = data[y, x]
    isnan(center) && return (NaN, 0)
    best  = -Inf
    steps = 0
    for r in 1:n_steps
        sx = x + round(Int, r * dcol)
        sy = y + round(Int, r * drow)
        (1 <= sx <= cols && 1 <= sy <= rows) || break
        @inbounds elev = data[sy, sx]
        isnan(elev) && break
        dist  = hypot((sx - x) * cellsize_x, (sy - y) * cellsize_y)
        best  = max(best, atan(elev - center, dist))
        steps = r
    end
    return steps == 0 ? (NaN, 0) : (Float64(best), steps)
end


In [ ]:
#| export
"""
    _openness_core(dem, cellsize_x, cellsize_y; radius_m, n_directions=8,
                   sign=1, nodata=nothing, min_coverage=1.0) -> Matrix{Float64}

Mean zenith angle (degrees) over `n_directions` ground-evenly-spaced azimuths,
searching to `radius_m` metres. `sign = 1` gives positive openness; `sign = -1`
gives negative openness (equivalent to running on `-dem`, without the copy).
`dem` must hold elevations (depths negative). A cell is reported only if every
ray completes at least `min_coverage` of its steps; otherwise it is NaN. PO and
NO therefore share an identical NaN footprint.
"""
function openness_core(dem::AbstractMatrix{T}, cellsize_x::Real, cellsize_y::Real;
                        radius_m::Real,
                        n_directions::Integer = 8,
                        sign::Integer = 1,
                        nodata = nothing,
                        min_coverage::Real = 1.0) where {T<:Real}
    radius_m > 0          || throw(ArgumentError("radius_m must be > 0, got $radius_m"))
    sign in (1, -1)       || throw(ArgumentError("sign must be 1 or -1, got $sign"))
    0 < min_coverage <= 1 || throw(ArgumentError("min_coverage must be in (0, 1], got $min_coverage"))

    dcol, drow, step_m = ray_directions(n_directions, cellsize_x, cellsize_y)  # validates n
    n_steps = [max(1, floor(Int, radius_m / s)) for s in step_m]                 # isotropic in metres

    rows, cols = size(dem)
    data = Matrix{Float64}(undef, rows, cols)
    @inbounds for i in eachindex(dem)
        v = dem[i]                                   # compare sentinel in source type
        data[i] = (nodata !== nothing && v == nodata) ? NaN : sign * Float64(v)
    end

    out = fill(NaN, rows, cols)
    Threads.@threads for x in 1:cols                 # one column per task: no races
        for y in 1:rows                              # y fastest: column-major
            @inbounds isnan(data[y, x]) && continue
            acc  = 0.0
            keep = true
            for d in 1:n_directions
                phi, done = max_elevation_angle(data, y, x, dcol[d], drow[d],
                                                 n_steps[d], cellsize_x, cellsize_y)
                if done < min_coverage * n_steps[d]
                    keep = false
                    break
                end
                acc += 90.0 - rad2deg(phi)
            end
            keep && (@inbounds out[y, x] = acc / n_directions)
        end
    end
    return out
end


In [ ]:
#| export
"""
positive_openness(dem; max_radius=10, n_directions=8, cellsize_x=1.0, cellsize_y=1.0) -> Matrix{Float64}

GA-SaMMT's "PO" (positive openness, section 2.1.1.2): the
"above-ground" view of the landscape. Well above 90 degrees on convex
forms (ridges, peaks, seamount tops); near 90 on flat/open terrain;
at or below 90 on enclosed/concave terrain.
"""
function positive_openness(dem::AbstractMatrix{<:Real};
                           max_radius::Real=10, n_directions::Integer=8,
                           cellsize_x::Real=1.0, cellsize_y::Real=1.0)
    return openness_core(dem, cellsize_x, cellsize_y; radius_m=max_radius, n_directions=n_directions)
end


In [ ]:
#| export
"""
negative_openness(dem; max_radius=10, n_directions=8, cellsize_x=1.0, cellsize_y=1.0) -> Matrix{Float64}

GA-SaMMT's "NO" (negative openness, section 2.1.1.2): the
"below-ground" view of the landscape. Well above 90 degrees on concave
forms (valleys, pits, depressions); near 90 on flat/open terrain; at
or below 90 on convex terrain. Computed as `positive_openness` applied
to the negated DEM -- see file header for why that's the correct,
standard definition and not just a shortcut.
"""
function negative_openness(dem::AbstractMatrix{<:Real};
                           max_radius::Real=10, n_directions::Integer=8,
                           cellsize_x::Real=1.0, cellsize_y::Real=1.0)
    return openness_core(-Float64.(dem), cellsize_x, cellsize_y; radius_m=max_radius, n_directions=n_directions)
end
